# Real PyG GNN Models in GNN Explorer

This demo shows `GNNVisualizer.add_model()` with real `torch_geometric.nn` layers: `GATConv`, `SAGEConv`, and `GINConv`. Each model is trained briefly on the built-in KarateClub graph, then rendered through the widget so users can inspect the message-passing layer and classifier output.

If the imports below fail, install the runtime packages in the notebook environment first:

```bash
python3 -m pip install torch torch-geometric
```

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GATConv, GINConv, SAGEConv

from gnn_exp import GNNVisualizer

In [ ]:
torch.manual_seed(7)

dataset = KarateClub()
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

display(Markdown(f"KarateClub has **{data.num_nodes} nodes**, **{data.edge_index.size(1)} directed edges**, **{num_features} input features**, and **{num_classes} classes**."))

In [ ]:
class GATNodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=2, concat=True)
        self.act1 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels * 2, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        return self.softmax(self.classifier(h))


class GraphSAGENodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        return self.softmax(self.classifier(h))


class GINNodeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(nn.Linear(in_channels, hidden_channels), nn.Tanh(), nn.Linear(hidden_channels, hidden_channels)))
        self.act1 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        h = self.act1(self.conv1(x, edge_index))
        return self.softmax(self.classifier(h))

In [ ]:
def train_model(model, data, epochs=120):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=5e-4)
    losses = []
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        probs = model(data.x, data.edge_index)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-9))[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))

    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
        train_acc = float((pred[data.train_mask] == data.y[data.train_mask]).float().mean())
        full_acc = float((pred == data.y).float().mean())

    return {"final_loss": losses[-1], "train_acc": train_acc, "full_acc": full_acc}


models = {
    "GAT": GATNodeClassifier(num_features, 4, num_classes),
    "GraphSAGE": GraphSAGENodeClassifier(num_features, 8, num_classes),
    "GIN": GINNodeClassifier(num_features, 8, num_classes),
}

assert isinstance(models["GAT"].conv1, GATConv)
assert isinstance(models["GraphSAGE"].conv1, SAGEConv)
assert isinstance(models["GIN"].conv1, GINConv)

results = {name: train_model(model, data) for name, model in models.items()}
rows = ["| Model | Real PyG layer | Final loss | Train acc | Full-graph acc |", "|---|---:|---:|---:|---:|"]
for name, model in models.items():
    metric = results[name]
    rows.append(f"| {name} | `{type(model.conv1).__name__}` | {metric['final_loss']:.4f} | {metric['train_acc']:.2f} | {metric['full_acc']:.2f} |")
display(Markdown("\n".join(rows)))

The next cells render each trained model. Click a node feature in the first hidden layer to expand the GNN operation. GAT is rendered with basic projection support; exact per-edge/per-head attention coefficients are not displayed yet.

In [ ]:
def make_visualizer(model, query_pair):
    visualizer = GNNVisualizer(renderer="svg")
    visualizer.add_model(
        data=data,
        model=model,
        subgraphSample=True,
        queries=[query_pair],
        mode="node",
    )
    return visualizer


visualizers = {
    "GAT": make_visualizer(models["GAT"], [0, 1]),
    "GraphSAGE": make_visualizer(models["GraphSAGE"], [0, 1]),
    "GIN": make_visualizer(models["GIN"], [0, 1]),
}

summary_rows = ["| Model | Visualized layer type | Aggregation label |", "|---|---:|---:|"]
for name, visualizer in visualizers.items():
    layer = visualizer.modelInfo["conv1"]
    summary_rows.append(f"| {name} | `{layer['type']}` | `{layer.get('aggregation')}` |")
display(Markdown("\n".join(summary_rows)))

## GATConv

In [ ]:
display(visualizers["GAT"])

## GraphSAGE / SAGEConv

In [ ]:
display(visualizers["GraphSAGE"])

## GINConv

In [ ]:
display(visualizers["GIN"])